# GIS Workshop 3: Cartography & Map Design
**Applied GIS & Spatial Analytics · Rutgers University Libraries · Fall 2026**

**Date:** Tuesday, October 20, 2026 · 4:00 – 6:00 PM
**Instructor:** Terri-ann Grant, GIS Data Specialist
**Contact:** terriann.grant@rutgers.edu

---

## What You Will Build

In Workshop 2 you produced a functional choropleth map of NJ internet access rates
and exported a GeoPackage. Today you take that map from *functional* to
*publication-quality* in six steps:

| Step | Topic | Key Skill |
|------|-------|-----------|
| 1 | Load Portfolio Data | Read a GeoPackage |
| 2 | Color Accessibility | Colorblind-safe palettes (cividis, YlOrBr) |
| 3 | Map Furniture | North arrow, scale bar, title, source credit |
| 4 | Inset Locator Map | NJ in northeast US context |
| 5 | Annotation & Labels | County callouts with data values |
| 6 | Print Export | PDF and 300 dpi PNG |

**Prerequisites:** Workshop 2 complete (or run the fallback in Step 1).
All code runs in Google Colab — no installation needed.

## Setup
Install `matplotlib-scalebar` (not in Colab by default) and import all libraries.

In [ ]:
# Install matplotlib-scalebar for metric scale bars
!pip install matplotlib-scalebar --quiet

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib_scalebar.scalebar import ScaleBar
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

print("✓ All libraries imported")
print("  GeoPandas:", gpd.__version__)
print("  Matplotlib:", plt.matplotlib.__version__)

---
## Step 1 — Load Portfolio Data

Load the GeoPackage exported at the end of Workshop 2.
If the file doesn't exist (you didn't export it, or the session reset),
the **fallback block** regenerates the same data from scratch.

In [ ]:
# ── Upload Workshop 2 GeoPackage ──────────────────────────────────────────
from google.colab import files
import os

print("Please select and upload 'nj_equity_counties.gpkg' from your computer:")
uploaded = files.upload()

for file_name in uploaded.keys():
    if file_name.endswith('.gpkg'):
        # Move or ensure it is saved as /content/nj_equity_counties.gpkg
        os.rename(file_name, "/content/nj_equity_counties.gpkg")
        print(f"\n✓ Successfully uploaded and saved to /content/nj_equity_counties.gpkg")
    else:
        print(f"\n⚠ Warning: You uploaded {file_name}. Please make sure to upload a .gpkg file.")

In [ ]:
# ── Try to load Workshop 2 GeoPackage ──────────────────────────────────
gpkg_path = Path("/content/nj_equity_counties.gpkg")

if gpkg_path.exists():
    nj = gpd.read_file(gpkg_path, layer="nj_equity_counties")
    print(f"✓ Loaded from GeoPackage: {nj.shape[0]} rows")
else:
    print("GeoPackage not found — running fallback (≈30 sec)...")

    # Fallback: rebuild from Census + equity CSV
    # ── County boundaries ──
    tiger_url = (
        "https://www2.census.gov/geo/tiger/TIGER2022/COUNTY/"
        "tl_2022_us_county.zip"
    )
    counties = gpd.read_file(tiger_url)
    nj_bounds = counties[counties["STATEFP"] == "34"].copy()
    nj_bounds = nj_bounds.to_crs(epsg=32618)
    nj_bounds["NAME"] = nj_bounds["NAME"].str.strip()

    # ── Equity data ──
    equity_url = (
        "https://raw.githubusercontent.com/rutgers-lib/gis-workshop-data/"
        "main/nj_digital_equity_2022.csv"
    )
    try:
        equity = pd.read_csv(equity_url)
    except Exception:
        # Local fallback: construct synthetic equity data
        print("  (Using synthetic equity data for demonstration)")
        county_names = [
            "Atlantic","Bergen","Burlington","Camden","Cape May",
            "Cumberland","Essex","Gloucester","Hudson","Hunterdon",
            "Mercer","Middlesex","Monmouth","Morris","Ocean",
            "Passaic","Salem","Somerset","Sussex","Union","Warren"
        ]
        rng = np.random.default_rng(42)
        rates = rng.uniform(40, 92, len(county_names))
        rates[county_names.index("Salem")] = np.nan      # intentional NaN
        rates[county_names.index("Cumberland")] = 39.0   # lowest
        rates[county_names.index("Hunterdon")] = 91.8    # highest
        equity = pd.DataFrame({
            "county": county_names,
            "internet_access_rate": rates,
        })

    equity["county"] = equity["county"].str.strip()

    # ── Attribute join ──
    nj = nj_bounds.merge(equity, left_on="NAME", right_on="county", how="left")
    nj["area_km2"] = nj.geometry.area / 1e6
    print(f"✓ Fallback complete: {nj.shape[0]} rows")

# ── Verify ──────────────────────────────────────────────────────────────
print(f"CRS: {nj.crs}")
print(f"Columns: {nj.columns.tolist()}")
print(f"\nSample data:")
display(nj[["NAME", "internet_access_rate", "area_km2"]].sort_values("NAME"))

> **Check:** You should see 21 rows and `EPSG:32618` confirmed above.
> Salem should show `NaN` for `internet_access_rate` — that's intentional.

---
## Step 2 — Color Accessibility

The `YlGnBu` palette from Workshop 2 is reasonable but has issues in
deuteranopia (red-green color blindness). Let's compare it to two
colorblind-safe alternatives.

### Why color matters
- **Jet / Rainbow**: Perceptually non-uniform — creates false visual boundaries
- **YlGnBu**: OK for most, problematic at the green-yellow boundary for deuteranopia
- **cividis**: Perceptually uniform, colorblind-safe, greyscale-readable
- **YlOrBr**: Sequential ColorBrewer palette, excellent for access/density maps

In [ ]:
# ── Side-by-side palette comparison ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 9))

palettes = [
    ("YlGnBu (Workshop 2)", "YlGnBu"),
    ("cividis (colorblind-safe)", "cividis"),
    ("YlOrBr (ColorBrewer safe)", "YlOrBr"),
]

for ax, (title, cmap) in zip(axes, palettes):
    nj.plot(
        column="internet_access_rate",
        cmap=cmap,
        scheme="quantiles",
        k=5,
        legend=True,
        legend_kwds={
            "title": "Internet Access Rate (%)",
            "loc": "upper center",
            "bbox_to_anchor": (0.5, -0.05),
            "fontsize": 9,
            "title_fontsize": 10,
            "frameon": False
        },
        missing_kwds={"color": "lightgrey", "hatch": "////",
                     "label": "No data (Salem)"},
        edgecolor="white",
        linewidth=0.6,
        ax=ax,
    )
    ax.set_title(title, fontsize=13, fontweight="bold", pad=10)
    ax.set_axis_off()

plt.suptitle("NJ Internet Access Rate — Palette Comparison",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("/content/palette_comparison.png", dpi=150,
            bbox_inches="tight", facecolor="white")
plt.show()
print("\n✓ Comparison saved to /content/palette_comparison.png")

### Exercise 1 — Try a Different Palette

The cell below plots a single map. Change `cmap` to any of the following
colorblind-safe options and observe how the pattern appears:

- `"viridis"` — dark purple to yellow, perceptually uniform
- `"plasma"` — purple to orange to yellow
- `"Blues"` — simple sequential blue
- `"BuPu"` — blue to purple (good for density)

> **Reflection (add a text cell):** Which palette best communicates which
> counties need investment? Does your answer change if you imagine a greyscale
> printout?

In [ ]:
# ── Exercise 1: Change the cmap value below ─────────────────────────────
cmap_choice = "cividis"        # ← change this value

fig, ax = plt.subplots(figsize=(8, 10))
nj.plot(
    column="internet_access_rate",
    cmap=cmap_choice,
    scheme="quantiles",
    k=5,
    legend=True,
    legend_kwds={"title": "Internet Access Rate (%)", "loc": "lower right"},

    missing_kwds={"color": "lightgrey", "hatch": "////",
                 "label": "No data (Salem)"},
    edgecolor="white",
    linewidth=0.6,
    ax=ax,
)
ax.set_title(f"NJ Internet Access Rate ({cmap_choice})", fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()
print(f"Palette used: {cmap_choice}")

---
## Step 3 — Map Furniture

Every publication map needs five elements:

| Element | What it does |
|---------|-------------|
| **Title block** | States what the map shows and for whom |
| **Legend** | Maps every color class to a data range |
| **North arrow** | Shows map orientation |
| **Scale bar** | Shows real-world distance (requires projected CRS) |
| **Source credit** | Names the data, vintage, and map author |

We build all five on a single map.

In [ ]:
#Define add north arrow function

def add_north_arrow(ax, x=0.95, y=0.92, size=0.055):
        """Draw a north arrow on axes using ax.transAxes coordinates."""
        ax.annotate(
            "N",
            xy=(x, y),
            xytext=(x, y - size),
            xycoords=ax.transAxes,
            textcoords=ax.transAxes,
            arrowprops=dict(
                facecolor="black",
                edgecolor="black",
                width=2.5,
                headwidth=8,
                headlength=6,
            ),
            ha="center",
            va="center",
            fontsize=11,
            fontweight="bold",
            color="black",
        )


    # ── Build the furnished map ─────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(10, 12))

    # Choropleth with cividis
nj.plot(
        column="internet_access_rate",
        cmap="cividis",
        scheme="quantiles",
        k=5,
        legend=True,
        legend_kwds={
            "title": "Internet Access Rate (%)",
            "loc": "lower right",
            "framealpha": 0.9,
        },
        missing_kwds={
            "color": "lightgrey",
            "hatch": "////",
            "label": "No data (Salem Co.)",
        },
        edgecolor="white",
        linewidth=0.7,
        ax=ax,
    )

    # Title block
ax.set_title("Internet Access Rate by County New Jersey, 2022",
        fontsize=16,
        fontweight="bold",
        loc="left",
        pad=12,
    )

    # Scale bar (requires EPSG:32618 — units in meters)
scalebar = ScaleBar(
        1,                           # 1 unit = 1 meter (EPSG:32618)
        units="m",
        dimension="si-length",
        length_fraction=0.25,
        location="lower left",
        box_alpha=0.85,
        color="black",
        font_properties={"size": 9},
    )
ax.add_artist(scalebar)

    # North arrow
add_north_arrow(ax, x=0.94, y=0.94)

    # Source credit
ax.annotate(
        "Data: US Census Bureau ACS 2022 · TIGER/Line 2022 · Map: T. Grant, Rutgers University",
        xy=(0.01, -0.03),
        xycoords="axes fraction",
        fontsize=7,
        color="#555555",
        va="bottom",
    )

ax.set_axis_off()
plt.tight_layout()
plt.savefig("/content/nj_equity_furnished.png", dpi=200,
                bbox_inches="tight", facecolor="white")
plt.show()
print("✓ Furnished map saved to /content/nj_equity_furnished.png")

### Exercise 2 — Customize the Furniture

Try at least one of the following:

1. **Move the north arrow** — change `x` and `y` in `add_north_arrow()` (both are 0–1, as fractions of the axes)
2. **Change scale bar location** — use `"upper left"`, `"upper right"`, or `"lower right"`
3. **Change the title** — write a title as a takeaway statement, e.g. *"Rural Counties Have the Lowest Internet Access"*
4. **Add a subtitle** — use `ax.text()` to place smaller text below the title

> **Reflection:** Does the position of the north arrow and scale bar affect
> how you read the map? Where do your eyes go first?

---
## Step 4 — Inset Locator Map

An inset locator map shows **where New Jersey is** within a larger geographic
context. This is a professional standard for any regional map — it orients
readers who may not immediately recognize New Jersey's outline.

### Approach
1. Download northeast US state boundaries (Census TIGER)
2. Create a sub-axes inside the main figure using `inset_axes()`
3. Plot all northeast states in grey
4. Highlight New Jersey in gold

In [ ]:
# ── Load northeast states ────────────────────────────────────────────────
print("Loading northeast state boundaries...")
states_url = (
    "https://www2.census.gov/geo/tiger/GENZ2022/shp/"
    "cb_2022_us_state_500k.zip"
)
states = gpd.read_file(states_url)
states = states.to_crs(epsg=4326)  # Geographic CRS for multi-state display

# Filter to northeastern states
ne_states = [
    "New Jersey", "New York", "Pennsylvania", "Delaware", "Maryland",
    "Connecticut", "Rhode Island", "Massachusetts", "Vermont",
    "New Hampshire", "Maine",
]
northeast = states[states["NAME"].isin(ne_states)].copy()
print(f"✓ Loaded {len(northeast)} northeast states")
print(northeast["NAME"].tolist())

In [ ]:
# ── Build map with inset ─────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(12, 12))

# Main choropleth
nj.plot(
    column="internet_access_rate",
    cmap="cividis",
    scheme="quantiles",
    k=5,
    legend=True,
    legend_kwds={
        "title": "Internet Access Rate (%)",
        "loc": "lower right",
        "framealpha": 0.9,
    },
    missing_kwds={
        "color": "lightgrey",
        "hatch": "////",
        "label": "No data (Salem Co.)",
    },
    edgecolor="white",
    linewidth=0.7,
    ax=ax,
)

# Update legend labels to replace commas with dashes
# We get the legend object from the axes after plotting
leg = ax.get_legend()
if leg is not None:
    for text in leg.get_texts():
        text.set_text(text.get_text().replace(", ", " - "))

# Map furniture
ax.set_title("Internet Access Rate by County New Jersey, 2022",
    fontsize=16, fontweight="bold", loc="left", pad=12,
)
scalebar = ScaleBar(1, units="m", dimension="si-length",
                    length_fraction=0.25, location="lower left",
                    box_alpha=0.85, color="black",
                    font_properties={"size": 9})
ax.add_artist(scalebar)
add_north_arrow(ax, x=0.94, y=0.94)
ax.annotate(
    "Data: US Census Bureau ACS 2022 · TIGER/Line 2022 · Map: T. Grant, Rutgers University",
    xy=(0.01, -0.03), xycoords="axes fraction",
    fontsize=7, color="#555555", va="bottom",
)
ax.set_axis_off()

# ── Inset locator map ───────────────────────────────────────────────────
# Positioned on the left side, nudged outward using bbox_to_anchor to avoid covering New Jersey
ax_inset = inset_axes(
    ax,
    width="30%",
    height="30%",
    loc="upper left",
    bbox_to_anchor=(-0.08, 0, 1, 1),
    bbox_transform=ax.transAxes,
    borderpad=1.0,
)

# All NE states in light grey
northeast.plot(
    ax=ax_inset,
    color="#D0D5DD",
    edgecolor="white",
    linewidth=0.5,
)

# Highlight New Jersey in gold
northeast[northeast["NAME"] == "New Jersey"].plot(
    ax=ax_inset,
    color="#CC8800",
    edgecolor="white",
    linewidth=0.8,
)

# Remove all ticks, labels, and axis coordinates
ax_inset.set_xticklabels([])
ax_inset.set_yticklabels([])
ax_inset.xaxis.get_label().set_visible(False)
ax_inset.yaxis.get_label().set_visible(False)
ax_inset.tick_params(axis='both', which='both', length=0)
ax_inset.set_title("NJ in Context", fontsize=8, pad=5, color="#333333")

# Explicitly force a thin border around the inset
for spine in ax_inset.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor("#888888")
    spine.set_linewidth(1.0)

plt.tight_layout()
plt.savefig("/content/nj_equity_with_inset.png", dpi=200,
            bbox_inches="tight", facecolor="white")
plt.show()
print("✓ Map with inset saved to /content/nj_equity_with_inset.png")

> **What you just built:** `inset_axes()` creates a sub-axes at a fixed
> position and size relative to the parent axes. We plot on it just like any
> other axes — the only difference is we use EPSG:4326 (geographic) for the
> context layer since we're showing multiple states at a coarse scale.

> **Note on CRS:** The inset uses a different CRS from the main map (4326 vs 32618).
> That's intentional — the slight distortion in the overview is fine at this scale,
> and it avoids projecting the full national state file to a NJ-specific projection.

---
## Step 5 — Annotation & Labels

Annotation means choosing **which counties to label** and **what to say**
about them. For a map about digital equity, the natural choices are:

- **Hunterdon** (91.8%) — highest access county
- **Cumberland** (39.0%) — lowest access county
- **One or two more** — counties mentioned in the accompanying text

### Key technique: `path_effects` for legibility
White halos around dark text (and dark halos around light text) keep labels
readable regardless of the map color underneath.

In [ ]:
# ── Build annotated map ─────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(10, 12))

# Main choropleth
nj.plot(
    column="internet_access_rate",
    cmap="cividis",
    scheme="quantiles",
    k=5,
    legend=True,
    legend_kwds={
        "title": "Internet Access Rate (%)",
        "loc": "lower right",
        "framealpha": 0.9,
    },
    missing_kwds={
        "color": "lightgrey",
        "hatch": "////",
        "label": "No data (Salem Co.)",
    },
    edgecolor="white",
    linewidth=0.7,
    ax=ax,
)

# Update legend labels to replace commas with dashes
leg = ax.get_legend()
if leg is not None:
    for text in leg.get_texts():
        text.set_text(text.get_text().replace(", ", " - "))

# Map furniture (Title and Subtitle)
ax.set_title("Internet Access Rate by County New Jersey, 2022",
    fontsize=16, fontweight="bold", loc="left", pad=24,
)
ax.text(
    0.0, 1.01,
    "Highlighting Highest and Lowest access rate",
    fontsize=11, fontstyle="italic", color="#444444",
    transform=ax.transAxes, ha="left", va="bottom"
)

scalebar = ScaleBar(1, units="m", dimension="si-length",
                    length_fraction=0.25, location="lower left",
                    box_alpha=0.85, color="black",
                    font_properties={"size": 9})
ax.add_artist(scalebar)
add_north_arrow(ax, x=0.94, y=0.94)
ax.annotate(
    "Data: US Census Bureau ACS 2022 · TIGER/Line 2022 · Map: T. Grant, Rutgers University",
    xy=(0.01, 0.01), xycoords="axes fraction",
    fontsize=7, color="#555555", va="bottom",
)
ax.set_axis_off()

# Inset
ax_inset = inset_axes(
    ax,
    width="30%",
    height="30%",
    loc="upper left",
    bbox_to_anchor=(-0.08, 0, 1, 1),
    bbox_transform=ax.transAxes,
    borderpad=1.0,
)
northeast.plot(ax=ax_inset, color="#D0D5DD", edgecolor="white", linewidth=0.5)
northeast[northeast["NAME"] == "New Jersey"].plot(
    ax=ax_inset, color="#CC8800", edgecolor="white", linewidth=0.8)

# Clean inset axis
ax_inset.set_xticklabels([])
ax_inset.set_yticklabels([])
ax_inset.xaxis.get_label().set_visible(False)
ax_inset.yaxis.get_label().set_visible(False)
ax_inset.tick_params(axis='both', which='both', length=0)
ax_inset.set_title("NJ in Context", fontsize=8, pad=5, color="#333333")

# Force border around inset
for spine in ax_inset.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor("#888888")
    spine.set_linewidth(1.0)

# ── Annotation ──────────────────────────────────────────────────────────
# Select counties to label: top 2 + bottom 1 (skip NaN)
valid = nj.dropna(subset=["internet_access_rate"])
label_rows = pd.concat([
    valid.nlargest(2, "internet_access_rate"),
    valid.nsmallest(2, "internet_access_rate"),
])

for _, row in label_rows.iterrows():
    centroid = row.geometry.centroid
    rate = row["internet_access_rate"]
    label = f"{row['NAME']}\n{rate:.1f}%"

    ax.annotate(
        label,
        xy=(centroid.x, centroid.y),
        ha="center",
        va="center",
        fontsize=8.5,
        fontweight="bold",
        color="white",
        path_effects=[
            pe.withStroke(linewidth=2.5, foreground="black"),
        ],
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="#1A2B4A",
            alpha=0.75,
            edgecolor="none",
        ),
    )

plt.tight_layout()
plt.savefig("/content/nj_equity_annotated.png", dpi=200,
            bbox_inches="tight", facecolor="white")
plt.show()

print("\nAnnotated counties:")
for _, row in label_rows.iterrows():
    print(f"  {row['NAME']:15s}  {row['internet_access_rate']:.1f}%")

### Exercise 3 — Add Your Own Annotation

Choose **one more county** to annotate. Ideas:
- A county you live in or near
- The county you think is most surprising in the data
- Essex County (Newark) — New Jersey's most populous

Add it by extending the `label_rows` DataFrame or by calling `ax.annotate()`
directly with the county's centroid coordinates.

> **Tip:** Get a specific county's centroid:
> ```python
> county = nj[nj["NAME"] == "Essex"]
> cx, cy = county.geometry.centroid.x.values[0], county.geometry.centroid.y.values[0]
> ```

---
## Step 6 — Print-Quality Export

| Format | DPI | Best for |
|--------|-----|---------|
| PNG | 300 | Slide decks, email, digital reports |
| PDF | — | Print, academic reports, any size |
| SVG | — | Editable in Illustrator/Inkscape |

The `bbox_inches="tight"` parameter trims whitespace automatically.
`facecolor="white"` prevents transparent backgrounds (which appear black
on some slide backgrounds).

In [ ]:
# ── Build final publication-quality map ──────────────────────────────────

if 'northeast' not in dir():
    raise RuntimeError("Run Step 4 (cells 15–16) before this cell.")
fig, ax = plt.subplots(1, 1, figsize=(10, 12))

nj.plot(
    column="internet_access_rate",
    cmap="cividis",
    scheme="quantiles",
    k=5,
    legend=True,
    legend_kwds={
        "title": "Internet Access Rate (%)",
        "loc": "lower right",
        "framealpha": 0.9,
    },
    missing_kwds={
        "color": "lightgrey",
        "hatch": "////",
        "label": "No data (Salem Co.)",
    },
    edgecolor="white",
    linewidth=0.7,
    ax=ax,
)

# Update legend labels to replace commas with dashes
leg = ax.get_legend()
if leg is not None:
    for text in leg.get_texts():
        text.set_text(text.get_text().replace(", ", " - "))

# Map furniture (Title and Subtitle)
ax.set_title("Internet Access Rate by County New Jersey, 2022",
    fontsize=16, fontweight="bold", loc="left", pad=24,
)
ax.text(
    0.0, 1.01,
    "Counties with highest and Lowest access rate",
    fontsize=11, fontstyle="italic", color="#444444",
    transform=ax.transAxes, ha="left", va="bottom"
)

scalebar = ScaleBar(1, units="m", dimension="si-length",
                    length_fraction=0.25, location="lower left",
                    box_alpha=0.85, color="black",
                    font_properties={"size": 9})
ax.add_artist(scalebar)
add_north_arrow(ax, x=0.94, y=0.94)
ax.annotate(
    "Data: US Census Bureau ACS 2022 · TIGER/Line 2022 · Map: T. Grant, Rutgers University",
    xy=(0.01, -0.05), xycoords="axes fraction",
    fontsize=7, color="#555555", va="bottom",
)
ax.set_axis_off()

# Inset
ax_inset = inset_axes(
    ax,
    width="30%",
    height="30%",
    loc="upper left",
    bbox_to_anchor=(-0.08, 0, 1, 1),
    bbox_transform=ax.transAxes,
    borderpad=1.0,
)
northeast.plot(ax=ax_inset, color="#D0D5DD", edgecolor="white", linewidth=0.5)
northeast[northeast["NAME"] == "New Jersey"].plot(
    ax=ax_inset, color="#CC8800", edgecolor="white", linewidth=0.8)

# Clean inset axis
ax_inset.set_xticklabels([])
ax_inset.set_yticklabels([])
ax_inset.xaxis.get_label().set_visible(False)
ax_inset.yaxis.get_label().set_visible(False)
ax_inset.tick_params(axis='both', which='both', length=0)
ax_inset.set_title("NJ in Context", fontsize=8, pad=5, color="#333333")

# Force border around inset
for spine in ax_inset.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor("#888888")
    spine.set_linewidth(1.0)

# Annotation
valid = nj.dropna(subset=["internet_access_rate"])
label_rows = pd.concat([
    valid.nlargest(2, "internet_access_rate"),
    valid.nsmallest(2, "internet_access_rate"),
])
for _, row in label_rows.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        f"{row['NAME']}\n{row['internet_access_rate']:.1f}%",
        xy=(centroid.x, centroid.y),
        ha="center", va="center",
        fontsize=8.5, fontweight="bold", color="white",
        path_effects=[pe.withStroke(linewidth=2.5, foreground="black")],
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#1A2B4A",
                  alpha=0.75, edgecolor="none"),
    )

# ── Export ──────────────────────────────────────────────────────────────
# High-resolution PNG for slide decks and digital use
fig.savefig("/content/nj_equity_map_300dpi.png",
            dpi=300, bbox_inches="tight", facecolor="white")
print("✓ Saved: nj_equity_map_300dpi.png (300 dpi PNG)")

# PDF for print — vector, infinitely scalable
fig.savefig("/content/nj_equity_map.pdf",
            dpi=300, bbox_inches="tight", facecolor="white")
print("✓ Saved: nj_equity_map.pdf (vector PDF)")

plt.show()

In [ ]:
# ── Download both files ──────────────────────────────────────────────────
from google.colab import files

print("Downloading files to your computer...")
files.download("/content/nj_equity_map_300dpi.png")
files.download("/content/nj_equity_map.pdf")
print("✓ Download triggered — check your browser's download folder")
print()
print("Portfolio files:")
import os
for f in ["/content/nj_equity_map_300dpi.png", "/content/nj_equity_map.pdf"]:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {f.split('/')[-1]:35s}  {size_kb:.0f} KB")

---
## Stretch Challenge — Design for a Non-Technical Audience

Take your final map and adapt it for an executive audience:

1. **Simplify the legend** — reduce to 3 classes (`k=3`) with plain labels:
   "Low Access", "Moderate Access", "High Access"
2. **Rewrite the title** as a clear takeaway:
   *"Six Counties Have Internet Access Rates Below 60%"*
3. **Reduce annotation** — label only the single lowest-access county
4. **Increase font sizes** — title 20pt, legend 12pt (readable on a phone screen)

Compare your executive brief to the technical map above. What information
is gained by simplifying? What is lost?

> **Portfolio reflection:** Add a Markdown cell below your map comparing
> the two versions. Which one would you send to a policymaker? Which to a
> researcher?

In [ ]:
# ── Stretch: Executive brief map ────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(10, 12))

nj.plot(
    column="internet_access_rate",
    cmap="cividis",
    scheme="quantiles",
    k=3,                        # ← simplified to 3 classes
    legend=True,
    legend_kwds={
        "title": "Internet Access Rate",
        "loc": "lower right",
        "framealpha": 0.9,
    },
    missing_kwds={
        "color": "lightgrey",
        "hatch": "////",
        "label": "No data",
    },
    edgecolor="white",
    linewidth=0.7,
    ax=ax,
)

# Update legend labels to replace commas with dashes
leg = ax.get_legend()
if leg is not None:
    for text in leg.get_texts():
        text.set_text(text.get_text().replace(", ", " - "))

# ── Change this title to a takeaway statement ──
ax.set_title(
    "Six Counties in New Jersey Have Internet Access\nRates Below 60% — 2022",
    fontsize=18, fontweight="bold", loc="left", pad=12,
)

scalebar = ScaleBar(1, units="m", dimension="si-length",
                    length_fraction=0.25, location="lower left",
                    box_alpha=0.85, color="black",
                    font_properties={"size": 9})
ax.add_artist(scalebar)
add_north_arrow(ax, x=0.94, y=0.94)
ax.annotate(
    "Data: ACS 2022 · Map: T. Grant, Rutgers University",
    xy=(0.01, -0.03), xycoords="axes fraction",
    fontsize=7, color="#555555", va="bottom",
)
ax.set_axis_off()

# Label only the lowest county
lowest = nj.dropna(subset=["internet_access_rate"]).nsmallest(1, "internet_access_rate")
for _, row in lowest.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        f"{row['NAME']}\n{row['internet_access_rate']:.1f}%\n(lowest)",
        xy=(centroid.x, centroid.y),
        ha="center", va="center",
        fontsize=10, fontweight="bold", color="white",
        path_effects=[pe.withStroke(linewidth=3, foreground="black")],
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#C0392B",
                  alpha=0.85, edgecolor="none"),
    )

plt.tight_layout()
plt.savefig("/content/nj_equity_executive_brief.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("✓ Executive brief saved to /content/nj_equity_executive_brief.png")

---
## Reflection

Add a Markdown cell here and answer at least two of the following:

1. **Color choice:** Which palette — YlGnBu, cividis, or YlOrBr — best
   communicates which counties need attention? Why?

2. **Furniture:** Of the five map furniture elements (title, legend, north arrow,
   scale bar, source), which one made the biggest difference to how you read the map?

3. **Inset:** Did the inset locator map change how you interpreted the county data?
   Who would benefit most from it?

4. **Annotation:** You labeled the highest and lowest counties. Is that the right
   choice? Are there other counties that would better support a policy argument?

5. **Audience:** What would you change about this map if it was going into a
   press release versus an academic report?

---
## Portfolio Summary

Today's outputs:

| File | Description |
|------|-------------|
| `palette_comparison.png` | Side-by-side of YlGnBu, cividis, YlOrBr |
| `nj_equity_furnished.png` | Map with complete furniture (no inset) |
| `nj_equity_with_inset.png` | Map with inset locator |
| `nj_equity_annotated.png` | Map with county callout labels |
| `nj_equity_map_300dpi.png` | **Portfolio deliverable** — 300 dpi PNG |
| `nj_equity_map.pdf` | **Portfolio deliverable** — vector PDF |
| `nj_equity_executive_brief.png` | Stretch: audience-adapted map |

### Save your GeoPackage for Workshop 4
Workshop 4 (GIS & Machine Learning, November 3) will use the same
`nj_equity_counties.gpkg` as input. Make sure it is saved to your Google Drive.

In [ ]:
# ── Verify portfolio files ───────────────────────────────────────────────
import os

portfolio_files = [
    "/content/nj_equity_map_300dpi.png",
    "/content/nj_equity_map.pdf",
]

print("Portfolio deliverables:")
for f in portfolio_files:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f"  ✓ {f.split('/')[-1]:35s} {size:6.0f} KB")
    else:
        print(f"  ✗ MISSING: {f}")

print()
print("All /content/ files:")
for f in sorted(os.listdir("/content/")):
    if f.endswith((".png", ".pdf", ".gpkg", ".ipynb")):
        size = os.path.getsize(f"/content/{f}") / 1024
        print(f"  {f:40s} {size:6.0f} KB")

---
## Next Session

**Workshop 4: GIS & Machine Learning** — Monday, November 3, 2026

Topics:
- Spatial autocorrelation (Moran's I)
- Feature engineering for spatial data
- Predictive modeling of digital equity outcomes
- Interpreting spatial model results

**What to bring:** Your `nj_equity_counties.gpkg` from today.

Register: libraries.rutgers.edu/events